In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1. 讀入深度學習套件

In [ ]:
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.datasets import imdb

### 2. 讀入數據

一般自然語言處理, 我們會限制最大要使用的字數。

In [ ]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

In [ ]:
print(f'訓練資料筆數：{len(x_train)}')
print(f'測試資料筆數：{len(x_test)}')

訓練資料筆數：25000
測試資料筆數：25000


注意每筆評論的長度當然是不一樣的。

In [ ]:
print(f'第一筆訓練資料的長度：{len(x_train[0])}')
print(f'第二筆測試資料的長度：{len(x_train[1])}')

第一筆訓練資料的長度：218
第二筆測試資料的長度：189


In [ ]:
print(f'第一筆資料的標籤：{y_train[0]}(正評)')
print(f'第二筆資料的標籤：{y_train[1]}(負評)')

第一筆資料的標籤：1(正評)
第二筆資料的標籤：0(負評)


### 3. 資料處理

雖然我們可以做真的 seq2seq, 可是資料長度不一樣對計算上有麻煩, 因此平常還是會固定一定長度, 其餘補 0。

In [ ]:
x_train = sequence.pad_sequences(x_train, maxlen=100)
x_test = sequence.pad_sequences(x_test, maxlen=100)

### 4. step 01: 打造一個函數學習機

In [ ]:
model = Sequential(Embedding(10000, 128)，LSTM(128)，Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',
             optimizer='adam',
             metrics=['accuracy'])

model.summary()
#LSTM para = （１２８＋１２８）*４*１２８　＋　１２８*４　＝　１３１５８４

#### 欣賞我們的 model

### 5. step 02: 訓練

In [ ]:
model.fit(x_train, y_train, batch_size=32, epochs=10,
          validation_data=(x_test, y_test))

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 173s 222ms/step - accuracy: 0.8951 - loss: 0.2659 - val_accuracy: 0.8512 - val_loss: 0.3501
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 174s 222ms/step - accuracy: 0.9354 - loss: 0.1752 - val_accuracy: 0.8486 - val_loss: 0.3725
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 174s 222ms/step - accuracy: 0.9582 - loss: 0.1179 - val_accuracy: 0.8342 - val_loss: 0.4418
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 201s 222ms/step - accuracy: 0.9721 - loss: 0.0853 - val_accuracy: 0.8431 - val_loss: 0.5053
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 203s 223ms/step - accuracy: 0.9792 - loss: 0.0613 - val_accuracy: 0.8338 - val_loss: 0.6447
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 201s 222ms/step - accuracy: 0.9882 - loss: 0.0410 - val_accuracy: 0.8341 - val_loss: 0.6490
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 176s 225ms/step - accuracy: 0.9909 - loss: 0.0312 - val_accuracy: 0.8232 - val_loss: 0.8202
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 208s 233ms/step - accuracy: 0.9907 -

### 6. 換個存檔方式

這次是把 model 和訓練權重分開存, 使用上更有彈性。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd '/content/drive/My Drive/Colab Notebooks'

/content/drive/My Drive/Colab Notebooks


In [ ]:
model_json = model.to_json()
open('imdb_model_architecture.json', 'w').write(model_json)
model.save_weights('imdb_model_weights.weights.h5') # Added '.weights' to the filename extension

test

In [ ]:
import numpy as np

# 定義計算 IMDB 分數的函數
def imdb_score(text, num_words=10000):  # Assuming your vocabulary size is 10000
    # 將文字分詞，並將每個詞轉換為對應的索引, 處理未知單詞，並限制在詞彙表範圍內
    seq = [word_index.get(word, 0) for word in text.split()]
    seq = [index for index in seq if index < num_words]  # Keep indices within vocabulary range
    seq_array = np.array([seq])  # 將列表 seq 轉換為 NumPy 陣列
    predictions = model.predict(seq_array)  # 使用模型進行預測
    score = predictions[0][0]  # 提取預測結果
    return score

# 讓使用者輸入文字
user_input = input("Please enter your comment: ")

# 計算並輸出分數
score = imdb_score(user_input)
print(f"Predicted score: {score}")

Please enter your comment:  Christopher Nolan’s ‘Inception’ offers a complex, thought-provoking story with strong performances and impressive visuals. Its intricate plot may be challenging but provides an engaging experience.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Predicted score: 0.9707570672035217
